# Quickbuild B - Run Multiple Flood Scenarios

This notebook takes the setup generated by `A_setup_paramaribo.ipynb`, runs the model for multiple flood extremes, and compares key totals across scenarios.

In [6]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from d_health import load_run_config, run_model_from_toml, write_run_config_from_setup

settings_toml = Path("data") / "setup_paramaribo" / "settings.toml"
flood_dir = Path("data") / "flood_extremes"
runs_root = Path("outputs") / "run"

if not settings_toml.exists():
    raise FileNotFoundError(
        f"Missing setup file: {settings_toml}. Run A_setup_paramaribo.ipynb first."
    )

runs_root.mkdir(parents=True, exist_ok=True)
settings_toml

WindowsPath('data/setup_paramaribo/settings.toml')

In [7]:
scenario_levels = [22, 1, 2, 3, 5]
rows = []

for wl in scenario_levels:
    wl_label = f"{wl:g}".replace(".", "p")
    scenario_name = f"scenario_wl{wl_label}m"
    scenario_dir = runs_root / scenario_name
    scenario_dir.mkdir(parents=True, exist_ok=True)

    flood_map = flood_dir / f"flood_wl{wl_label}m_paramaribo.tif"
    if not flood_map.exists():
        raise FileNotFoundError(f"Missing flood scenario map: {flood_map}")

    run_config = scenario_dir / "config.toml"
    write_run_config_from_setup(
        settings_toml=settings_toml,
        flood_depth_map=flood_map,
        run_config_path=run_config,
        output_out_dir=scenario_dir,
    )

    cfg = load_run_config(run_config)
    outputs = run_model_from_toml(run_config)

    with rasterio.open(flood_map) as src:
        depth = src.read(1).astype(np.float32)
    flooded = depth > 0
    flooded_cells = int(flooded.sum())
    mean_depth_m = float(depth[flooded].mean()) if flooded_cells else 0.0

    conc = outputs.pathogen_conc
    conc_mask = np.isfinite(conc) & (conc > 0)
    mean_concentration = float(conc[conc_mask].mean()) if conc_mask.any() else 0.0

    row = {
        "scenario": scenario_name,
        "water_level_m": wl,
        "flooded_cells": flooded_cells,
        "mean_depth_m": mean_depth_m,
        "mean_concentration": mean_concentration,
    }
    row.update({k: float(v) for k, v in outputs.totals.items()})

    for group_name, dose_arr in outputs.doses.items():
        dose_mask = np.isfinite(dose_arr) & (dose_arr > 0)
        row[f"mean_dose_{group_name}"] = (
            float(dose_arr[dose_mask].mean()) if dose_mask.any() else 0.0
        )

    for group in cfg.settings.population_groups:
        thresholds = group.depth_thresholds
        for i, the in enumerate(thresholds):
            lower = the.min_depth
            upper = thresholds[i + 1].min_depth if i + 1 < len(thresholds) else np.inf
            if i + 1 < len(thresholds):
                band = (depth >= lower) & (depth < upper)
            else:
                band = depth >= lower
            row[f"cells_{group.name}_{the.name}"] = int(band.sum())

    rows.append(row)

summary = pd.DataFrame(rows).sort_values("water_level_m").reset_index(drop=True)
summary

d_health.config.loaders | INFO | Loading run config from outputs\run\scenario_wl22m\config.toml
d_health.config.loaders | INFO | Loading run config from outputs\run\scenario_wl22m\config.toml
d_health.model.pipeline | INFO | Starting d_health pipeline
d_health.geo | INFO | Aligned input (181, 205) and clipped target onto 181 × 205 grid (resampling=average)
d_health.geo | INFO | Aligned input (181, 205) and clipped target onto 181 × 205 grid (resampling=nearest)
d_health.model.inputs | INFO | Loaded inputs: flood (181, 205), population (181, 205) (groups: [np.str_('children'), np.str_('adults'), np.str_('total')]), urban_rural (181, 205)
d_health.model.emissions | INFO | urban_rural: 7255 of 37105 cells (19.6%) unclassified — applying nodata_sanitation='none' (factor 1.0)
d_health.model.pipeline | INFO | Total infected_adults = 29553
d_health.model.pipeline | INFO | Total infected_children = 9971
d_health.postprocessing.coverage | INFO | Total inhabitants:                 280870
d_healt

FileNotFoundError: Missing flood scenario map: data\flood_extremes\flood_wl1m_paramaribo.tif

In [ ]:
infected_cols = sorted([c for c in summary.columns if c.startswith("infected_")])
dose_cols = sorted([c for c in summary.columns if c.startswith("mean_dose_")])

diagnostic_cols = ["flooded_cells", "mean_depth_m", "mean_concentration", *dose_cols]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

summary.set_index("scenario")[infected_cols].plot(
    kind="bar",
    ax=axes[0],
    title="Total infected by flood scenario",
)
axes[0].set_xlabel("Scenario")
axes[0].set_ylabel("Total infected population")

normalized = summary.set_index("scenario")[diagnostic_cols].copy()
for col in diagnostic_cols:
    max_val = normalized[col].max()
    normalized[col] = normalized[col] / max_val if max_val > 0 else 0.0

normalized.plot(
    marker="o",
    ax=axes[1],
    title="Normalized diagnostics (relative to each metric max)",
)
axes[1].set_xlabel("Scenario")
axes[1].set_ylabel("Relative value (0-1)")

fig.tight_layout()
summary.to_csv(runs_root / "scenario_summary.csv", index=False)
summary